# [LAB 04] 데이터 다루기 - 8. 데이터 재구조화

## #01. 준비작업 

### 1. 라이브러리 참조 

In [145]:
from jussam import load_data
from pandas import DataFrame, pivot_table, melt

### 2. 샘플 데이터 가져오기 

In [146]:
origin = load_data("city_people")
origin

📚 서울, 인천, 부산에서 2005년, 2010년 2015년에 조사한 가상의 인구수 데이터(인덱스와 메타데이터 없음)


,도시,연도,인구,지역
0,서울,2015,9904312,수도권
1,서울,2010,9631482,수도권
2,서울,2005,9762546,수도권
3,부산,2015,3448737,경상권
4,부산,2010,3393191,경상권
5,부산,2005,3512547,경상권
6,인천,2015,2890451,수도권
7,인천,2010,2632035,수도권


## #02. 피벗 테이블

In [147]:
### 1. 기본 사용 방법 ( 도시와 연도에 따른 인구수 재배치 )

In [148]:
pivot1 = pivot_table(origin, 
                     index = '도시', 
                     columns = '연도', 
                     values = '인구')

print(type(pivot1))
pivot1 

<class 'pandas.DataFrame'>


연도,2005,2010,2015
도시,,,
부산,3512547.000,3393191.000,3448737.000
서울,9762546.000,9631482.000,9904312.000
인천,NaN,2632035.000,2890451.000


### 2. 중복 데이터의 집계 방법 지정하기 

In [149]:
pivot_table(origin, 
            index = '지역', 
            columns = '연도', 
            values = '인구', 
            aggfunc = 'sum')

연도,2005,2010,2015
지역,,,
경상권,3512547,3393191,3448737
수도권,9762546,12263517,12794763


### 3. 두 개 이상의 집계 함수 지정 

In [150]:
pivot_table(origin, 
            index = '지역', 
            columns = '연도', 
            values= '인구', 
            aggfunc= ['sum', 'mean'])

sum                            mean                        
연도      2005      2010      2015        2005        2010        2015
지역                                                                  
경상권  3512547   3393191   3448737 3512547.000 3393191.000 3448737.000
수도권  9762546  12263517  12794763 9762546.000 6131758.500 6397381.500

### 4. 복수 인덱스 지정 

In [151]:
pivot_table(origin, 
            index = ['지역', '연도'], 
            columns = '도시', 
            values= '인구', 
            aggfunc= ['mean', 'sum'])

mean                                 sum              \
도시                부산          서울          인천          부산          서울   
지역  연도                                                                 
경상권 2005 3512547.000         NaN         NaN 3512547.000         NaN   
    2010 3393191.000         NaN         NaN 3393191.000         NaN   
    2015 3448737.000         NaN         NaN 3448737.000         NaN   
수도권 2005         NaN 9762546.000         NaN         NaN 9762546.000   
    2010         NaN 9631482.000 2632035.000         NaN 9631482.000   
    2015         NaN 9904312.000 2890451.000         NaN 9904312.000   

                      
도시                인천  
지역  연도                
경상권 2005         NaN  
    2010         NaN  
    2015         NaN  
수도권 2005         NaN  
    2010 2632035.000  
    2015 2890451.000

## #03. Melt 

### 1. 샘플 피벗 테이블 생성 

In [152]:
pivot_df = pivot_table(origin, 
                       index = '연도', columns= '지역', 
                       values= '인구', aggfunc= 'mean')

pivot_df

지역,경상권,수도권
연도,,
2005,3512547.000,9762546.000
2010,3393191.000,6131758.500
2015,3448737.000,6397381.500


### 2. 피벗 테이블 분리 1단계 

In [153]:
pivot_df2 = pivot_df.reset_index()
pivot_df2 

지역,연도,경상권,수도권
0,2005,3512547.000,9762546.000
1,2010,3393191.000,6131758.500
2,2015,3448737.000,6397381.500


### 3. 피벗 테이블 분리 2단계: melt 처리 

In [154]:
mdf = melt(pivot_df2, id_vars=['연도'], 
           value_vars=['경상권', '수도권'])

mdf

,연도,지역,value
0,2005,경상권,3512547.000
1,2010,경상권,3393191.000
2,2015,경상권,3448737.000
3,2005,수도권,9762546.000
4,2010,수도권,6131758.500
5,2015,수도권,6397381.500


### 4. 피벗 테이블 분리시 필드 이름 지정하기 

In [155]:
mdf = melt(pivot_df2, id_vars=['연도'], 
           value_vars = ['경상권', '수도권'], 
           var_name='구분', value_name='인구수')

mdf

,연도,구분,인구수
0,2005,경상권,3512547.000
1,2010,경상권,3393191.000
2,2015,경상권,3448737.000
3,2005,수도권,9762546.000
4,2010,수도권,6131758.500
5,2015,수도권,6397381.500


## #04. Stack, UnStack 실습을 위한 준비 작업 

In [156]:
origin = load_data("body_size")
origin

📚 어느 학급 학생들의 이름, 성별, 키, 몸무게에 대한 가상의 데이터

    field    description
--  -------  -------------
 0  name     이름
 1  sex      성별
 2  height   키
 3  weight   몸무게



,sex,height,weight
name,,,
Lee,M,175,98.000
Park,F,167,48.000
Hong,M,180,NaN
Kim,F,162,55.000
Nam,M,172,85.000


## #05. Stack 

### 1. 기본 사용 방법 

In [157]:
st1 = origin.stack()
print(type(st1))
st1 

<class 'pandas.Series'>


name        
Lee   sex           M
      height      175
      weight   98.000
Park  sex           F
      height      167
      weight   48.000
Hong  sex           M
      height      180
      weight      NaN
Kim   sex           F
      height      162
      weight   55.000
Nam   sex           M
      height      172
      weight   85.000
dtype: object

### 2. Stack 결과를 DataFrame으로 생성하기 

In [158]:
df1 = DataFrame(origin.stack())
df1 

0
name              
Lee  sex         M
     height    175
     weight 98.000
Park sex         F
     height    167
     weight 48.000
Hong sex         M
     height    180
     weight    NaN
Kim  sex         F
     height    162
     weight 55.000
Nam  sex         M
     height    172
     weight 85.000

## #06. UnStack

### 1. Stack 결과를 원래대로 되돌린다.

In [159]:
st1.unstack()

,sex,height,weight
name,,,
Lee,M,175,98.000
Park,F,167,48.000
Hong,M,180,NaN
Kim,F,162,55.000
Nam,M,172,85.000


## #01. 연습문제 - 학생 흡연율 데이터 전처리 

In [160]:
from jussam import load_data

In [161]:
df = load_data('students_smoke')
df

📚 한 도시의 20개 중,고교를 대상으로 조사한 흡연율 자료 (인덱스와 메타데이터 없음)


,지역,구성,흡연율
0,도시,남,0.640
1,도시,여,0.450
2,농촌,공학,0.700
3,도시,남,0.850
4,농촌,남,0.720
5,도시,남,0.780
6,도시,여,0.620
7,도시,남,0.790
8,농촌,남,0.750
9,농촌,공학,0.810


In [162]:
df2 = pivot_table(df, 
            index = '지역', 
            columns= '구성', 
            values = '흡연율', 
            aggfunc= 'mean'
            )
df2

구성,공학,남,여
지역,,,
농촌,0.757,0.745,0.463
도시,0.680,0.792,0.507


In [163]:
df = df2.copy()
df

구성,공학,남,여
지역,,,
농촌,0.757,0.745,0.463
도시,0.680,0.792,0.507


## #02. 연습문제 - 동네 아이스크림 가게 매출 분석 

In [164]:
from jussam import load_data

In [165]:
df = load_data('icecream_sales')
df

📚 '달콤 스쿱' 아이스크림 가게의 매출 데이터 (인덱스 없음)


,Date,Flavor,Topping,Price,Quantity
0,2023-07-01,초콜릿,아몬드,3500,20.000
1,2023-07-01,바닐라,초코시럽,3000,25.000
2,2023-07-01,딸기,연유,3200,18.000
3,2023-07-02,민트초코,초코칩,3800,15.000
4,2023-07-02,초콜릿,아몬드,3500,22.000
5,2023-07-02,바닐라,NaN,3000,30.000
6,2023-07-03,딸기,연유,3200,25.000
7,2023-07-03,민트초코,초코칩,3800,NaN
8,2023-07-03,바닐라,초코시럽,3000,28.000
9,2023-07-04,초콜릿,아몬드,3500,18.000


In [166]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      12 non-null     datetime64[us]
 1   Flavor    12 non-null     str           
 2   Topping   11 non-null     str           
 3   Price     12 non-null     int64         
 4   Quantity  11 non-null     float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 818.0 bytes


In [167]:
df.isnull().sum()

Date        0
Flavor      0
Topping     1
Price       0
Quantity    1
dtype: int64

In [168]:
df2 = df.dropna()
df2

,Date,Flavor,Topping,Price,Quantity
0,2023-07-01,초콜릿,아몬드,3500,20.000
1,2023-07-01,바닐라,초코시럽,3000,25.000
2,2023-07-01,딸기,연유,3200,18.000
3,2023-07-02,민트초코,초코칩,3800,15.000
4,2023-07-02,초콜릿,아몬드,3500,22.000
6,2023-07-03,딸기,연유,3200,25.000
8,2023-07-03,바닐라,초코시럽,3000,28.000
9,2023-07-04,초콜릿,아몬드,3500,18.000
10,2023-07-04,딸기,연유,3200,22.000
11,2023-07-04,민트초코,초코칩,3800,17.000


In [169]:
pivot_df = pivot_table(df2, 
            index = 'Date', 
            columns= 'Flavor', 
            values='Quantity',
            aggfunc= 'mean'
            )
pivot_df

Flavor,딸기,민트초코,바닐라,초콜릿
Date,,,,
2023-07-01,18.000,NaN,25.000,20.000
2023-07-02,NaN,15.000,NaN,22.000
2023-07-03,25.000,NaN,28.000,NaN
2023-07-04,22.000,17.000,NaN,18.000


In [170]:
pivot_df = pivot_table(df2, 
            index = 'Date', 
            columns= 'Flavor', 
            values='Quantity',
            # aggfunc= 'mean'
            )
pivot_df

Flavor,딸기,민트초코,바닐라,초콜릿
Date,,,,
2023-07-01,18.000,NaN,25.000,20.000
2023-07-02,NaN,15.000,NaN,22.000
2023-07-03,25.000,NaN,28.000,NaN
2023-07-04,22.000,17.000,NaN,18.000


In [171]:
pivot_df2 = pivot_df.reset_index()
pivot_df2

Flavor,Date,딸기,민트초코,바닐라,초콜릿
0,2023-07-01,18.000,NaN,25.000,20.000
1,2023-07-02,NaN,15.000,NaN,22.000
2,2023-07-03,25.000,NaN,28.000,NaN
3,2023-07-04,22.000,17.000,NaN,18.000


In [172]:
df3 = melt(pivot_df2, id_vars='Date')
df3

,Date,Flavor,value
0,2023-07-01,딸기,18.000
1,2023-07-02,딸기,NaN
2,2023-07-03,딸기,25.000
3,2023-07-04,딸기,22.000
4,2023-07-01,민트초코,NaN
5,2023-07-02,민트초코,15.000
6,2023-07-03,민트초코,NaN
7,2023-07-04,민트초코,17.000
8,2023-07-01,바닐라,25.000
9,2023-07-02,바닐라,NaN


### 강사님 코드

In [174]:
df3 = melt(pivot_df2, id_vars=['Date'],
           value_vars=['딸기','민트초코','바닐라','초콜릿'],
           var_name='Flavor', value_name='Quantity')
df3

,Date,Flavor,Quantity
0,2023-07-01,딸기,18.000
1,2023-07-02,딸기,NaN
2,2023-07-03,딸기,25.000
3,2023-07-04,딸기,22.000
4,2023-07-01,민트초코,NaN
5,2023-07-02,민트초코,15.000
6,2023-07-03,민트초코,NaN
7,2023-07-04,민트초코,17.000
8,2023-07-01,바닐라,25.000
9,2023-07-02,바닐라,NaN


In [ ]:
# 응, 근데 사실 둘 다 같은 결과가 나와.
# value_vars, var_name, value_name 생략하면 pandas가 알아서 처리하거든:

# value_vars 생략 → id_vars 제외한 나머지 컬럼 전부 자동으로 씀
# var_name 생략 → 컬럼명이 "variable"로 기본값 설정
# value_name 생략 → 값 컬럼명이 "value"로 기본값 설정

# 그래서 차이는 컬럼 이름뿐이야:
# 코드맛 컬럼명수량 컬럼명네 코드variablevalue강사님 코드FlavorQuantity

# 문제 조건 보면 "컬럼 이름은 원본 이름을 그대로 유지해야 한다" 고 되어 있는데, 이건 맛 이름(딸기, 민트초코 등)을 말하는 거라 var_name/value_name 지정 여부랑은 별개야.
# 다만 시험이나 제출용이라면 강사님 코드처럼 명시적으로 쓰는 게 안전해. 채점자 입장에서 컬럼명이 Flavor, Quantity로 나오길 기대할 수 있으니까

In [ ]:
# melt 파라미터 정리해줄게.

# pd.melt() 파라미터 전체
# pythonpd.melt(
#     frame,          # 대상 DataFrame
#     id_vars,        # 고정할 컬럼 (녹이지 않을 것)
#     value_vars,     # 녹일 컬럼들 (생략 시 id_vars 제외 전부)
#     var_name,       # 녹인 컬럼명들이 들어갈 컬럼의 이름 (기본값: "variable")
#     value_name,     # 값들이 들어갈 컬럼의 이름 (기본값: "value")
#     col_level,      # 멀티인덱스 컬럼일 때만 사용 (거의 안 씀)
#     ignore_index    # True면 인덱스 0부터 재설정 (기본값: True)
# )

# 안 빼먹는 방법
# melt 쓸 때 이 순서로 생각해:

# "뭘 고정? → 뭘 녹여? → 녹인 결과 컬럼 이름은?"


# id_vars → 고정할 컬럼 (날짜, ID 같은 것)
# value_vars → 녹일 컬럼 (명시 안 해도 되지만 명시하는 게 안전)
# var_name → 컬럼명들이 값으로 들어갈 컬럼 이름
# value_name → 실제 수치가 들어갈 컬럼 이름

# var_name, value_name 안 쓰면 결과에 variable, value라는 이름 없는 컬럼이 생기는데, 나중에 다른 분석이나 merge할 때 헷갈려서 습관적으로 명시하는 게 좋아.

## #03. 연습문제 - 게임 토너먼트 최고 득점자 

In [175]:
from jussam import load_data

In [176]:
df = load_data('game_scores')
df    

📚 최근 열린 게임 토너먼트 경기 결과 데이터 (인덱스와 메타데이터 없음)


,player,stage1,stage2,stage3
0,Alice,85,92.000,88.000
1,Bob,78,NaN,95.000
2,Charlie,90,88.000,91.000
3,David,82,85.000,NaN
4,Eva,95,98.000,99.000


In [177]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   player  5 non-null      str    
 1   stage1  5 non-null      int64  
 2   stage2  4 non-null      float64
 3   stage3  4 non-null      float64
dtypes: float64(2), int64(1), str(1)
memory usage: 315.0 bytes


In [178]:
df.isnull().sum()

player    0
stage1    0
stage2    1
stage3    1
dtype: int64

In [179]:
df2 = df.dropna()
df2

,player,stage1,stage2,stage3
0,Alice,85,92.000,88.000
2,Charlie,90,88.000,91.000
4,Eva,95,98.000,99.000


In [ ]:
# player별로 보고 싶으니까, ser_index('playter')

In [185]:
df3 = df2.set_index('player')
df3

,stage1,stage2,stage3
player,,,
Alice,85,92.000,88.000
Charlie,90,88.000,91.000
Eva,95,98.000,99.000


In [ ]:
# 컬럼에 있는 stage1,2,3를 각 인덱스안에 넣고, 

In [186]:
df4 = DataFrame(df3.stack())
df4

0
player               
Alice   stage1 85.000
        stage2 92.000
        stage3 88.000
Charlie stage1 90.000
        stage2 88.000
        stage3 91.000
Eva     stage1 95.000
        stage2 98.000
        stage3 99.000

In [189]:
df5 = df4.reset_index()
df5

,player,level_1,0
0,Alice,stage1,85.000
1,Alice,stage2,92.000
2,Alice,stage3,88.000
3,Charlie,stage1,90.000
4,Charlie,stage2,88.000
5,Charlie,stage3,91.000
6,Eva,stage1,95.000
7,Eva,stage2,98.000
8,Eva,stage3,99.000


In [190]:
df5.filter(['player', 0]).groupby('player').mean()

,0
player,
Alice,88.333
Charlie,89.667
Eva,97.333


In [191]:
pivot_table('df2', 
            index ='player', 
            columns='stage1', 
            values='stage1',
            aggfunc='mean')

KeyError: 'stage1'

In [ ]:
st1 = df2.stack()
print(type(st1))
st1

<class 'pandas.Series'>


0  player      Alice
   stage1         85
   stage2     92.000
   stage3     88.000
2  player    Charlie
   stage1         90
   stage2     88.000
   stage3     91.000
4  player        Eva
   stage1         95
   stage2     98.000
   stage3     99.000
dtype: object